In [208]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import pickle
from sklearn.preprocessing import MinMaxScaler

sys.path.append('../')
from third_party.HDCmodel import *
from conformal_inference.models import *
from conformal_inference.methods import *
from conformal_inference.utils import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [247]:
class ConformalHDC():
    def __init__(self, inputs, labels, calib_size=0.2, 
                 dimension=10000, epochs=2, total_level=100,
                 sim_measure="cosine",
                 random_state=0, verbose=True, progress=True):

        self.calib_size = calib_size
        self.sim_measure = sim_measure

        self.random_state = random_state
        self.verbose = verbose
        self.progress= progress 

        inputs_train, inputs_calib, labels_train, labels_calib =\
        train_test_split(inputs, labels, test_size=self.calib_size, random_state=self.random_state)

        # Train the HD model
        HDC = HyperDimensionalComputing(dimension, totalPos = inputs_train.shape[1], totalLevel = total_level, datatype = np.int16, buffer = [-1.0, 1.0], cuda = False)
        labels_train, labels_calib = genLabel(labels_train), genLabel(labels_calib)
        classHV = dict([(x, np.array([0 for _ in range(dimension)])) for x in range(1, len(np.unique(labels_train)) + 1)])
        baseVector = HDC.genBaseVector(HDC.P, -1, HDC.dim)
        levelVector = HDC.genLevelVector(HDC.Q, -1, HDC.dim)
        
        HVector = HV_encoding(HDC, baseVector, levelVector, inputs_train)
        HVector_calib = HV_encoding(HDC, baseVector, levelVector, inputs_calib)
        self.classHVs = HDC.genClassHV(classHV, labels_train, HVector)
        self.classHVs, _ = HDC.oneShotTraining(self.classHVs, HVector, labels_train, HVector, labels_train)
        self.classHVs, _, _, _ = HDC.retraining(self.classHVs, HVector, labels_train, HVector, labels_train, epochs)

        # store model for future use
        self.HDC = HDC
        self.baseVector = baseVector
        self.levelVector = levelVector
        self.labels = np.unique(labels)
    
        # Compute the calibration scores
        self.scores_calib = {}
        scores_calib = self._compute_similarity_scores(HVector_calib, labels_calib)

        for i, label in enumerate(labels_calib):
            if (label in self.scores_calib.keys()):
                self.scores_calib[label].append(scores_calib[i])
            else:
                self.scores_calib[label] = [scores_calib[i]]

    
    
    # implement the encoding functions and other similarity scores later below 


    def _compute_similarity_scores(self, HVs, labels):
        ''' Computes the similarity scores of the HVs and corresponding classHVs
        '''

        classHVs_batch =  np.array([self.classHVs[label] for label in labels])

        if self.sim_measure == "euclidean":
            scores = np.linalg.norm(HVs - classHVs_batch, axis=1)
        elif self.sim_measure == "cosine":
            # Compute the dot product for each pair of vectors
            dot_product = np.einsum('ij,ij->i', HVs, classHVs_batch)
            norm_HV1 = np.linalg.norm(HVs, axis=1)
            norm_HV2 = np.linalg.norm(classHVs_batch, axis=1)
            scores = -dot_product / (norm_HV1 * norm_HV2 + 1e-10)
        return scores


    def predict(self, inputs_test):
        scores = []
        n_test = len(inputs_test)
        for label in self.labels:
            tmp_scores = self._compute_similarity_scores(inputs_test, [label]*n_test).reshape(-1,1)
            scores.append(tmp_scores)
        
        scores = np.concatenate(scores,axis=1)
        # if self.sim_measure == "euclidean":
        #     predicted_indices = np.argmin(scores, axis=1)
        # elif self.sim_measute == "cosine":
        #     predicted_indices = np.argmax(scores, axis=1)
        predicted_indices = np.argmin(scores, axis=1)
        predictions = [self.labels[idx] for idx in predicted_indices]

        return predictions


    def conformalPS(self, inputs_test, alpha, allow_empty=False):
        ''' Computes the conformal prediction sets at significance level alpha
        '''
        n_test = len(inputs_test)
        inputs_test = HV_encoding(self.HDC, self.baseVector, self.levelVector, inputs_test)
        self.quantiles = {}

        psets = psets = [[] for _ in range(n_test)] 
        for label in self.labels:
            n_calib = len(self.scores_calib[label])
            scores = self._compute_similarity_scores(inputs_test, [label]*n_test).reshape(-1,1)
            quantile = np.quantile(self.scores_calib[label], (n_calib+1)*(1-alpha)/n_calib)
            self.quantiles[label] = quantile

            for i in range(n_test):
                if scores[i] < quantile:
                    psets[i].append(label)
        
        if not allow_empty:
            pred = self.predict(inputs_test)
            for i, pset in enumerate(psets):
                if len(pset)==0:
                    pset.append(pred[i])

        return psets

In [300]:
def run_experiment(random_state):
    base_path="C:/Users/liang/Documents/GitHub/conformalHDC/data/CTG"
    df= pd.read_csv(os.path.join(base_path, 'cleaned_data.csv'))
    X = np.array(df.drop(columns=['Label']))
    y = np.array(df['Label'], dtype=int)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=random_state)
    np.random.seed(random_state)
    
    # benchmark
    dimension = 10000
    epochs = 4
    HDC = HyperDimensionalComputing(dimension, totalPos = X_train.shape[1], totalLevel = 100, datatype = np.int16, buffer = [-1.0, 1.0], cuda = False)
    labels_train, labels_test = genLabel(y_train), genLabel(y_test)
    classHV = dict([(x, np.array([0 for _ in range(dimension)])) for x in range(1, len(np.unique(labels_train)) + 1)])
    baseVector = HDC.genBaseVector(HDC.P, -1, HDC.dim)
    levelVector = HDC.genLevelVector(HDC.Q, -1, HDC.dim)
    
    HVector = HV_encoding(HDC, baseVector, levelVector, X_train)
    HVector_test = HV_encoding(HDC, baseVector, levelVector, X_test)
    classHVs = HDC.genClassHV(classHV, labels_train, HVector)
    classHVs, _ = HDC.oneShotTraining(classHVs, HVector, labels_train, HVector, labels_train)
    classHVs, _, _, _ = HDC.retraining(classHVs, HVector, labels_train, HVector, labels_train, epochs)

    preds=HDC.predict(classHVs, HVector_test,labels_test)
    preds = np.array(preds).reshape(-1, 1)

    # Evaluate the predictions (hdc method)
    df_m_pred = eval_m_psets(preds, y_test)
    df_m_pred['method'] = 'HDC'
    
    df_lc_pred = eval_lc_psets(preds, y_test)
    df_lc_pred['method'] = 'HDC'

    # conformalHDC
    alpha = 0.1
    conformal = ConformalHDC(X_train, y_train, random_state=random_state, epochs=epochs, dimension=dimension, total_level=100)
    pset = conformal.conformalPS(X_test, alpha, allow_empty=True)

    # Evaluate the conformal prediction sets (conformal method)
    df_m_pset = eval_m_psets(pset, y_test)
    df_m_pset['method'] = 'conformalHDC'
    
    df_lc_pset = eval_lc_psets(pset, y_test)
    df_lc_pset['method'] = 'conformalHDC'

    # Concatenate results for this run
    result_m = pd.concat([df_m_pset, df_m_pred], ignore_index=True)
    result_lc = pd.concat([df_lc_pset, df_lc_pred], ignore_index=True)
    result_m['random_state'] = random_state
    result_lc['random_state'] = random_state
    
    return result_m, result_lc

In [307]:
# Initialize empty lists to store results
all_results_m = []
all_results_lc = []

repetitions=50
# Loop to run 50 repetitions with different random seeds
for i in tqdm(range(repetitions)):
    random_state = i
    result_m, result_lc = run_experiment(random_state)
    
    # Append the results to the list
    all_results_m.append(result_m)
    all_results_lc.append(result_lc)

# Concatenate all the results from all repetitions and variance settings
final_m_df = pd.concat(all_results_m, ignore_index=True)
final_lc_df = pd.concat(all_results_lc, ignore_index=True)

  0%|                                                                                           | 0/50 [00:00<?, ?it/s]

Epoch 1 accuracy: 93.88395190799791
Epoch 2 accuracy: 95.45216936748562
Epoch 3 accuracy: 96.44537375849451
Epoch 4 accuracy: 97.54312598013591
Epoch 1 accuracy: 94.05228758169935
Epoch 2 accuracy: 95.68627450980392
Epoch 3 accuracy: 96.5359477124183
Epoch 4 accuracy: 97.25490196078431


  2%|█▋                                                                                 | 1/50 [00:40<33:07, 40.55s/it]

Epoch 1 accuracy: 93.62258233141662
Epoch 2 accuracy: 95.71353894406691
Epoch 3 accuracy: 96.49764767381077
Epoch 4 accuracy: 97.0726607422896
Epoch 1 accuracy: 94.11764705882352
Epoch 2 accuracy: 95.68627450980392
Epoch 3 accuracy: 96.73202614379085
Epoch 4 accuracy: 97.12418300653594


  4%|███▎                                                                               | 2/50 [01:22<33:03, 41.33s/it]

...

 96%|██████████████████████████████████████████████████████████████████████████████▋   | 48/50 [32:10<01:18, 39.27s/it]

Epoch 1 accuracy: 93.46576058546785
Epoch 2 accuracy: 95.03397804495557
Epoch 3 accuracy: 96.60219550444327
Epoch 4 accuracy: 97.17720857292211
Epoch 1 accuracy: 93.0718954248366
Epoch 2 accuracy: 94.50980392156862
Epoch 3 accuracy: 96.01307189542484
Epoch 4 accuracy: 96.66666666666667


 98%|████████████████████████████████████████████████████████████████████████████████▎ | 49/50 [32:49<00:39, 39.08s/it]

Epoch 1 accuracy: 93.72713016204914
Epoch 2 accuracy: 95.66126502875065
Epoch 3 accuracy: 96.39309984317825
Epoch 4 accuracy: 97.17720857292211
Epoch 1 accuracy: 93.92156862745098
Epoch 2 accuracy: 95.68627450980392
Epoch 3 accuracy: 97.12418300653594
Epoch 4 accuracy: 97.45098039215686


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [33:27<00:00, 40.15s/it]


In [308]:
final_m_df

,M-coverage,M-size,M-size|cov,method,random_state
0,0.873239,1.708920,1.924731,conformalHDC,0
1,0.934272,1.000000,1.000000,HDC,0
2,0.892019,1.807512,1.926316,conformalHDC,1
3,0.910798,1.000000,1.000000,HDC,1
4,0.915493,1.835681,1.933333,conformalHDC,2
...,...,...,...,...,...
95,0.934272,1.000000,1.000000,HDC,47
96,0.896714,1.774648,1.910995,conformalHDC,48
97,0.981221,1.000000,1.000000,HDC,48
98,0.906103,1.826291,1.943005,conformalHDC,49


In [309]:
final_lc_df

,class1-LC-coverage,class1-LC-size,class1-LC-size|cov,class2-LC-coverage,class2-LC-size,class2-LC-size|cov,method,random_state
0,0.884146,1.804878,2.0,0.836735,1.387755,1.658537,conformalHDC,0
1,0.957317,1.000000,1.0,0.857143,1.000000,1.000000,HDC,0
2,0.876543,1.870370,2.0,0.941176,1.607843,1.708333,conformalHDC,1
3,0.944444,1.000000,1.0,0.803922,1.000000,1.000000,HDC,1
4,0.911765,1.905882,2.0,0.930233,1.558140,1.675000,conformalHDC,2
...,...,...,...,...,...,...,...,...
95,0.963190,1.000000,1.0,0.840000,1.000000,1.000000,HDC,47
96,0.905325,1.887574,2.0,0.863636,1.340909,1.552632,conformalHDC,48
97,0.982249,1.000000,1.0,0.977273,1.000000,1.000000,HDC,48
98,0.886228,1.856287,2.0,0.978261,1.717391,1.755556,conformalHDC,49


In [310]:
# Define a function to calculate the standard error
def standard_error(x):
    return np.std(x, ddof=1) / np.sqrt(len(x))

# Group by 'method' and calculate mean and standard error for each relevant column
aggregated_df = final_lc_df.groupby('method').agg({
    'class1-LC-coverage': ['mean', standard_error],
    'class1-LC-size': ['mean', standard_error],
    'class1-LC-size|cov': ['mean', standard_error],
    'class2-LC-coverage': ['mean', standard_error],
    'class2-LC-size': ['mean', standard_error],
    'class2-LC-size|cov': ['mean', standard_error]
}).reset_index()

# Rename columns for clarity
aggregated_df.columns = ['_'.join(col).strip() if col[1] else col[0] for col in aggregated_df.columns.values]

print(aggregated_df)

         method  class1-LC-coverage_mean  class1-LC-coverage_standard_error  \
0           HDC                 0.965423                           0.002006   
1  conformalHDC                 0.893356                           0.003585   

   class1-LC-size_mean  class1-LC-size_standard_error  \
0             1.000000                       0.000000   
1             1.852044                       0.006297   

   class1-LC-size|cov_mean  class1-LC-size|cov_standard_error  \
0                  1.00000                           0.000000   
1                  1.99972                           0.000196   

   class2-LC-coverage_mean  class2-LC-coverage_standard_error  \
0                 0.846217                           0.006883   
1                 0.900655                           0.007092   

   class2-LC-size_mean  class2-LC-size_standard_error  \
0             1.000000                       0.000000   
1             1.477099                       0.016346   

   class2-LC-size|cov_mean